# 🧪 Thực Nghiệm Độc Lập: Toàn Bộ 5 Bài Test Khoa Học Chứng Minh Điểm Yếu Của LiDAR Trên Google Colab
### Khung Lý Thuyết: Dimension-Free Lipschitz Bound & Randomized Smoothing (RS-LiDAR)

Notebook này là môi trường hoàn chỉnh thực thi **Toàn Bộ 5 Bài Test Khoa Học**:
1. **Bài Test 1 (Solver Error Resilience & Khảo sát $\sigma$)**: So sánh sai số hình học giữa DPM-5 (lookahead) và DDIM-50 (chuẩn), đo sai số reward trên 5 mô hình (ImageReward, CLIP, HPS v2.1, Aesthetic Score, PickScore) và khảo sát $\sigma \in \{0.05, 0.10, 0.15, 0.25, 0.50, 1.00\}$.
2. **Bài Test 2 (Softmax Mode Collapse / Entropy)**: Đo độ sụp đổ Entropy Shannon $H(w^r)$, chứng minh hiện tượng Best-of-1 Trap do hệ số $\lambda = 5000$.
3. **Bài Test 3 (Guidance Vector Field Stability)**: Đo độ ổn định góc quay Cosine $\text{CosSim}(\mathbf{g}_t, \mathbf{g}_{t+\delta})$ khi có nhiễu vi mô $\delta = 10^{-3}$.
4. **Bài Test 4 (Effective Sample Size ESS & Particle Starvation)**: Đo số lượng hạt hữu hiệu $\text{ESS}_t = \frac{1}{\sum (w_i^r)^2}$, tỷ lệ hạt áp đảo $w_{\max}$ và số hạt hoạt động. Bóc trần sự lãng phí 98% chi phí tính toán đa hạt ở LiDAR gốc.
5. **Bài Test 5 (Step-Budget Solver Scaling & Theorem 1 Truncation Bound)**: Khảo sát các mốc bước $S \in \{2, 3, 5, 8, 15\}$, chứng minh RS-LiDAR tại $S=3$ đạt độ chính xác tương đương hoặc vượt trội LiDAR gốc tại $S=5$.

*Lưu ý*: Test 2, 3, 4 hoàn toàn là các phép tính ma trận đại số trên latent, chạy siêu tốc chỉ mất ~30 giây!

## 1. Kiểm Tra Phần Cứng GPU

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB")
else:
    raise RuntimeError("❌ Vui lòng chọn Runtime -> Change runtime type -> GPU!")
!nvidia-smi

## 2. Gắn Kết Google Drive & Thiết Lập Mã Nguồn

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
DRIVE_DIR = f"{base_drive}/RS-LiDAR/test_results"
os.makedirs(DRIVE_DIR, exist_ok=True)

REPO_DIR = "/content/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin main

WORKDIR = REPO_DIR if os.path.exists(f"{REPO_DIR}/test_lidar_weaknesses.py") else f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
%cd {WORKDIR}
print("✅ Thư mục làm việc hiện tại:", os.getcwd())

## 3. Cài Đặt Môi Trường Đầy Đủ

In [ ]:
import os, urllib.request

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

import hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)
print("✅ Môi trường cho Bộ 3 Bài Test đã hoàn tất!")

## 4. Chạy Toàn Bộ Hệ Thống 5 Bài Test Thực Nghiệm Khoa Học (`--test all`)
Cấu hình chuẩn khoa học: 20 prompts, 20 particles, khảo sát bán kính làm mịn $\sigma \in \{0.05, 0.10, 0.15, 0.25, 0.50, 1.00\}$ (mặc định $\sigma=0.05$), tích hợp đầy đủ Bộ 5 Mô Hình Reward (ImageReward, CLIP, HPS v2.1, Aesthetic Score, PickScore). Toàn bộ biểu đồ 6 panel, biểu đồ ablation đa mô hình và các bảng số liệu CSV/Markdown sẽ tự động lưu ra Google Drive.

In [ ]:
# Tự động kiểm tra và chuyển đến đúng thư mục chứa script (chống lỗi No such file or directory)
import os

possible_dirs = [
    "/content/RS-LiDAR/Diffusion-LiDAR-Sampling",
    "/content/RS-LiDAR",
    "Diffusion-LiDAR-Sampling",
    "."
]
found = False
for p in possible_dirs:
    if os.path.exists(os.path.join(p, "test_lidar_weaknesses.py")):
        %cd {p}
        found = True
        break

if not found and not os.path.exists("test_lidar_weaknesses.py"):
    print("⚡ Chưa tìm thấy repo, tiến hành tự động git clone...")
    !git clone https://github.com/leekwanreal/RS-LiDAR.git /content/RS-LiDAR
    %cd /content/RS-LiDAR/Diffusion-LiDAR-Sampling
else:
    print("⚡ Đã tìm thấy repo, tiến hành git pull để cập nhật mã nguồn mới nhất...")
    !git pull origin main

out_dir = DRIVE_DIR if ('DRIVE_DIR' in locals() and os.path.exists(os.path.dirname(DRIVE_DIR))) else './test_results'
print("🚀 Bắt đầu thực thi trọn bộ 5 bài test tại:", os.getcwd())

!python test_lidar_weaknesses.py \
    --test all \
    --num_prompts 20 \
    --num_particles 20 \
    --sigma 0.05 \
    --tune_sigma \
    --sigmas "0.05,0.10,0.15,0.25,0.50,1.00" \
    --all_rewards \
    --overwrite \
    --output_dir "{out_dir}"

## 5. Hiển Thị Toàn Bộ Bảng Kết Quả & Đồ Thị Tổng Hợp 5 Bài Test


In [ ]:
import pandas as pd, glob, os
from IPython.display import display, Image

# Tự động xác định thư mục kết quả (Google Drive hoặc thư mục cục bộ)
target_dir = out_dir if 'out_dir' in locals() else (DRIVE_DIR if 'DRIVE_DIR' in locals() and os.path.exists(DRIVE_DIR) else './test_results')
print(f"📂 Đang tải kết quả từ: {target_dir}\n")

# 1. Hiển thị Bảng tổng hợp khoa học 5 bài test
table_files = glob.glob(f"{target_dir}/*comparison*.csv")
if table_files:
    df = pd.read_csv(table_files[0])
    print("📊 [BẢNG 1] BẢNG TỔNG HỢP KHOA HỌC 5 BÀI TEST (WEAKNESSES COMPARISON):")
    display(df)
else:
    print("ℹ️ Chưa tìm thấy bảng weaknesses comparison table.")

# 2. Hiển thị Bảng khảo sát Ablation Sigma
sigma_files = glob.glob(f"{target_dir}/*sigma*.csv")
if sigma_files:
    df_sigma = pd.read_csv(sigma_files[0])
    print("\n📈 [BẢNG 2] BẢNG KHẢO SÁT BÁN KÍNH LÀM MỊN SIGMA (SIGMA ABLATION BENCHMARK):")
    display(df_sigma)
else:
    print("ℹ️ Chưa tìm thấy bảng sigma ablation table.")

# 3. Hiển thị tất cả các đồ thị biểu diễn (6-Panel Composite Plot & Multi-Reward Ablation Curves)
png_files = sorted(glob.glob(f"{target_dir}/*.png"))
if png_files:
    for p in png_files:
        print(f"\n🖼️ Đồ thị: {os.path.basename(p)}")
        display(Image(filename=p))
else:
    print("ℹ️ Chưa tìm thấy file ảnh đồ thị.")
